In [12]:
#Définition des outils
from langchain.tools import tool, ToolRuntime
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]
@tool

def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [14]:
#Création d’un agent HITL
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_ollama import ChatOllama

model = ChatOllama(
model="llama3.2:3b", # ou mistral, gemma, etc.
temperature=0
)

class EmailState(AgentState):
    email: str

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
},
description_prefix="Tool execution requires approval",
),
],
)

In [15]:
#Tester l’agent
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {
        "messages": [HumanMessage(content="Veuillez lire mon e-mail et envoyer une réponse immédiatement. Envoyez la réponse maintenant dans le même fil de discussion.")],
        "email": "Bonjour Sara, je vais être en retard pour notre réunion de demain. Pouvons-nous la reprogrammer ? Cordialement, Sofia"
},
config=config
)
print(response)

{'messages': [HumanMessage(content='Veuillez lire mon e-mail et envoyer une réponse immédiatement. Envoyez la réponse maintenant dans le même fil de discussion.', additional_kwargs={}, response_metadata={}, id='1deb33ae-b66c-42a9-a57f-824f011acc7b'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-07-06T21:45:14.4322404Z', 'done': True, 'done_reason': 'stop', 'total_duration': 48802556200, 'load_duration': 25565331500, 'prompt_eval_count': 218, 'prompt_eval_duration': 10239247000, 'eval_count': 34, 'eval_duration': 7557805000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019f3963-bceb-7c01-ad6a-796b3e677c2f-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '8e510960-c38b-45ad-80da-162dd16d4a98', 'type': 'tool_call'}, {'name': 'send_email', 'args': {'body': 'Réponse à votre message'}, 'id': 'ac507261-803d-45e7-8cf9-1aa3cdaff58c', 'type': 'tool_call'}], invalid_tool_calls=[], usag

In [6]:
#Afficher le message interrompu avec metadata
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Réponse à votre message'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Réponse à votre message'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='a34a37a2f61551b4879f4b658ce8412e')]


In [7]:
#Afficher seulement le message interrompu

print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Réponse à votre message


In [ ]:
#Approuver le résultat
from langgraph.types import Command
response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
),
    config=config # Le même thread ID pour reprendre la conversation
)

print(response['messages'][-1].content)

In [ ]:
#Refuser le résultat
response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                # Une explication sur les raisons du rejet
                    "message": " J’annule notre rendez-vous."
}
]
}
),
config=config # Le même thread ID pour reprendre la conversation
)
print(response)

In [ ]:
#Modifier le résultat
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        # Le nom du Tool.
                        "name": "send_email",

                        # Les arguments à passer au tool.
                        "args": {
                            "body": (
                                "Je suis désolée mais je dois annuler "
                                "notre rendez-vous je ne serais pas libre. Sara"
                            )
                        },
                    },
                }
            ]
        }
    ),
    config=config,  # Le même thread ID pour reprendre la conversation
)

print(response["messages"][-1].content)